In [1]:
!pip install earthengine-api geemap

In [4]:
import ee
import geemap

# Initialize directly without re-authenticating
try:
    ee.Initialize()
    print("Google Earth Engine is already connected!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Authenticated and connected successfully!")

Google Earth Engine is already connected!


In [9]:
import ee
import geemap

# Initialize GEE
ee.Initialize()

# 1. Load District/Administrative Boundaries for India (FAO GAUL dataset)
# Level 2 contains district boundaries (e.g., Gautam Buddha Nagar / Noida, Delhi, etc.)
boundaries = ee.FeatureCollection("FAO/GAUL/2015/level2")

# 2. Filter boundary by your district name (Change 'Gautam Buddha Nagar' or 'Delhi' to your district)
my_boundary = boundaries.filter(ee.Filter.eq('ADM2_NAME', 'Ranchi'))

# Create map centered on the district boundary
Map = geemap.Map()
Map.centerObject(my_boundary, 11)

# 3. Fetch Landsat 8 satellite imagery filtered by your boundary geometry
image = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
         .filterBounds(my_boundary)
         .filterDate('2023-05-01', '2023-07-31')
         .sort('CLOUD_COVER')
         .first())

# 4. Convert Thermal band to Celsius
thermal = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)

# 5. 👉 CLIP THERMAL LAYER TO EXACT BORDER 👈
thermal_clipped = thermal.clip(my_boundary)

# 6. Add visualization layers to map
vis_params = {'min': 25, 'max': 45, 'palette': ['blue', 'yellow', 'orange', 'red'], 'opacity':0.6}

# Add clipped heat map
Map.addLayer(thermal_clipped, vis_params, 'LST (°C) Clipped')

# Add black boundary outline so the border looks crisp
style_boundary = {'color': 'black', 'fillColor': '00000000', 'width': 2}
Map.addLayer(my_boundary.style(**style_boundary), {}, 'District Border')

# Display Map
Map

Map(center=[23.21122039036191, 85.33034703402323], controls=(WidgetControl(options=['position', 'transparent_b…

In [5]:
!pip install ipywidgets